# RecursiveMAS — build the WebLLM model in Colab

Builds a **latent-transfer-capable** WebLLM model (TVM + mlc-llm from source, pinned to
`v0.19.0` — the last release before the `apache-tvm-ffi` migration that breaks the current
wheels). Then patches the model to expose hidden states, compiles to a WebGPU `.wasm`, and
uploads to Hugging Face.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free). The GPU isn't needed
for the WebGPU compile (that's emcc codegen) but is needed for the optional RecursiveLink
training at the end. TVM build is ~25-35 min and stays built for the whole session, so you can
re-run the compile cell freely.


## 1. Check the runtime


In [ ]:
!nvidia-smi || echo 'no GPU (ok for compile; needed for training)'
!python --version


## 2. System build deps (LLVM + Polly + ninja + cmake)


In [ ]:
import subprocess, glob, os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

print("🔄 Clearing apt locks (Colab's unattended-upgrades holds them and apt hangs)...")
subprocess.run('pkill -9 -f unattended-upgrade 2>/dev/null || true', shell=True)
subprocess.run('rm -f /var/lib/dpkg/lock-frontend /var/lib/dpkg/lock '
               '/var/cache/apt/archives/lock /var/lib/apt/lists/lock; '
               'dpkg --configure -a 2>/dev/null || true', shell=True)

print("🔄 Updating packages...")
!apt-get update -qq

print("🔄 Installing build dependencies...")
!apt-get install -y -o Dpkg::Use-Pty=0 \
    cmake ninja-build build-essential git \
    llvm-dev libedit-dev libxml2-dev zlib1g-dev libssl-dev

# Get LLVM version and install Polly (TVM static-links LLVM incl. Polly; llvm-dev omits libPolly.a)
lv = subprocess.check_output(['llvm-config', '--version']).decode().split('.')[0].strip()
print(f'✅ LLVM major: {lv}')

!apt-get install -y -o Dpkg::Use-Pty=0 libpolly-{lv}-dev || apt-get install -y libpolly-18-dev || true

print('✅ libPolly.a:', (glob.glob('/usr/lib/**/libPolly.a', recursive=True) or ['MISSING'])[0])

## 3. Rust (for tokenizers) — force the reliable sparse crates.io index


In [ ]:
import os, shutil
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
# Non-interactive install (piped 'curl | sh' can hang in Colab). Use LATEST stable Rust:
# modern tokenizers deps (e.g. unicode-segmentation) require rustc >= 1.85. The v0.19
# deny-lint on old code is capped in the mlc-llm build cell via --cap-lints=allow.
if not shutil.which('rustup'):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs -o /tmp/rustup-init.sh
    !sh /tmp/rustup-init.sh -y --profile minimal --no-modify-path
!rustup default stable
os.environ['CARGO_REGISTRIES_CRATES_IO_PROTOCOL'] = 'sparse'
os.environ['CARGO_NET_GIT_FETCH_WITH_CLI'] = 'true'
!cargo --version

## 4. Emscripten (for the WebGPU `.wasm` target)


In [ ]:
%cd /content
!git clone https://github.com/emscripten-core/emsdk.git
!cd emsdk && ./emsdk install 3.1.56 && ./emsdk activate 3.1.56
# capture emsdk env into this Python process so emcc is on PATH for later cells
import subprocess, os
envtxt = subprocess.check_output('source /content/emsdk/emsdk_env.sh >/dev/null 2>&1 && env',
                                 shell=True, executable='/bin/bash').decode()
for line in envtxt.splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        if k == 'PATH' or k.startswith('EM') or k == 'EMSDK':
            os.environ[k] = v
!emcc --version | head -1


## 5. Clone mlc-llm `v0.19.0` (+ this repo for the scripts)


In [ ]:
%cd /content
!git clone --branch v0.19.0 --depth 1 --recursive --shallow-submodules https://github.com/mlc-ai/mlc-llm.git
!git clone --depth 1 https://github.com/vishalmysore/recursiveMASWebLLM.git


## 6. Build TVM from source  (~25-35 min, once per session)


In [ ]:
%%bash
set -e
cd /content/mlc-llm/3rdparty/tvm
mkdir -p build && cp cmake/config.cmake build/config.cmake
{
  echo 'set(CMAKE_BUILD_TYPE RelWithDebInfo)'
  echo 'set(USE_LLVM "llvm-config --link-static")'
  echo 'set(HIDE_PRIVATE_SYMBOLS ON)'
  echo 'set(USE_CUDA OFF)'; echo 'set(USE_METAL OFF)'
  echo 'set(USE_VULKAN OFF)'; echo 'set(USE_OPENCL OFF)'
} >> build/config.cmake
cd build && cmake .. -G Ninja && ninja


In [ ]:
import os
os.environ['TVM_HOME'] = '/content/mlc-llm/3rdparty/tvm'
os.environ['TVM_LIBRARY_PATH'] = '/content/mlc-llm/3rdparty/tvm/build'
os.environ['LD_LIBRARY_PATH'] = '/content/mlc-llm/3rdparty/tvm/build:' + os.environ.get('LD_LIBRARY_PATH', '')
!pip install -q -e /content/mlc-llm/3rdparty/tvm/python
!python -c "import tvm; print('TVM OK', tvm.__file__)"


## 7. Build mlc-llm  (~3-5 min)


In [ ]:
%%bash
set -e
cd /content/mlc-llm
# v0.19 tokenizers-c trips the deny-by-default 'dangerous_implicit_autorefs' lint on
# modern Rust; cap lints to allow for that crate's cargo build.
sed -i 's|set(TOKENIZERS_CPP_RUST_FLAGS "")|set(TOKENIZERS_CPP_RUST_FLAGS "--cap-lints=allow")|' 3rdparty/tokenizers-cpp/CMakeLists.txt
mkdir -p build
{
  echo 'set(CMAKE_BUILD_TYPE RelWithDebInfo)'
  echo 'set(USE_LLVM "llvm-config --link-static")'
  echo 'set(HIDE_PRIVATE_SYMBOLS ON)'
  echo 'set(TVM_SOURCE_DIR /content/mlc-llm/3rdparty/tvm)'
  echo 'set(USE_CUDA OFF)'; echo 'set(USE_METAL OFF)'
  echo 'set(USE_VULKAN OFF)'; echo 'set(USE_OPENCL OFF)'
} > build/config.cmake
cd build && cmake .. -G Ninja && ninja

In [ ]:
!pip install -q -e /content/mlc-llm/python
!pip install -q -U huggingface_hub
!python -c "import tvm, mlc_llm; print('mlc_llm OK', mlc_llm.__file__)"


## 8. Patch the model to expose hidden states (the RecursiveMAS bit)


In [ ]:
!python /content/recursiveMASWebLLM/expose_hidden.py --arch qwen2


## 9. Convert weights + compile to WebGPU `.wasm`
Re-run this cell freely while iterating — TVM/mlc-llm stay built.


In [ ]:
%%bash
set -e
# mlc_llm compile --device webgpu links mlc_wasm_runtime.bc -> build it first (emcc),
# and set MLC_LLM_SOURCE_DIR so the compiler can locate it.
export MLC_LLM_SOURCE_DIR=/content/mlc-llm
(cd /content/mlc-llm/web && TVM_SOURCE_DIR=/content/mlc-llm/3rdparty/tvm make dist/wasm/mlc_wasm_runtime.bc)
ls -la /content/mlc-llm/web/dist/wasm/
cd /content
MODEL=Qwen/Qwen2.5-0.5B-Instruct; NAME=RecursiveMAS-0.5B; QUANT=q4f16_1
hf download $MODEL --local-dir hf/$NAME
W=dist-model/$NAME-MLC; L=dist-model/libs; mkdir -p $W $L
mlc_llm convert_weight hf/$NAME --quantization $QUANT --model-type qwen2 -o $W
mlc_llm gen_config hf/$NAME --quantization $QUANT --model-type qwen2 \
    --conv-template qwen2 --prefill-chunk-size 1024 -o $W
mlc_llm compile $W/mlc-chat-config.json --device webgpu \
    -o $L/$NAME-$QUANT-webgpu.wasm
ls -la $L $W

## 10. Upload to Hugging Face
Get a **Write** token at https://huggingface.co/settings/tokens, then run the login cell and
paste it. The repo is created automatically on first upload.


In [ ]:
from huggingface_hub import login
login()  # paste your HF write token


In [ ]:
!hf upload vishalmysore/RecursiveMAS-0.5B-MLC dist-model/RecursiveMAS-0.5B-MLC .
# the .wasm is in dist-model/libs/ — download it and attach to a GitHub Release,
# or also push it: !hf upload vishalmysore/RecursiveMAS-0.5B-MLC dist-model/libs libs


## 11. (Optional) Train the RecursiveLink on the GPU
This is the step that actually benefits from Colab's GPU. Produces `recursivelink.json`.


In [ ]:
!pip install -q torch transformers
!python /content/recursiveMASWebLLM/train_recursivelink.py \
    --model Qwen/Qwen2.5-0.5B-Instruct --out dist-model/recursivelink.json
# then: !hf upload vishalmysore/RecursiveMAS-0.5B-MLC dist-model/recursivelink.json recursivelink.json


---
**Notes**
- Builds the same way as the repo's `.github/workflows/build-source.yml`, just interactive.
- The `.wasm` must match the app's `@mlc-ai/web-llm` version — pin both.
- You can't test WebGPU in Colab; load the model in the recursiveMAS web app via `CUSTOM_MODELS`.
